<div
     style="padding: 20px;
            color: white;
            font-size: 250%;
            text-align: center;
            display: fill;
            border-radius: 5px;
            background-color: #8be4a7ab;
            overflow: hidden;
            font-weight: 700;
            border: 5px solid #F28C28;"
     >
CNNs & Transfer Learning in Practice
</div>

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Setup & Loading the Data</b></p>
</div>

In [ ]:
# Install gdown if needed (Google Colab has it pre-installed)
# !pip install gdown

import gdown
import zipfile
import os

# Download pizza_steak dataset from Google Drive
gdown.download(id="1TwtxuFvaAVzkd-_EA6hPHps_jd2720pH", quiet=False)

# Extract the zip
with zipfile.ZipFile("/content/pizza_steak.zip", "r") as zip_ref:
    zip_ref.extractall("/content/")

print("Train classes:", os.listdir("/content/pizza_steak/train"))
print("Test classes:", os.listdir("/content/pizza_steak/test"))

# Count images per split
for split in ["train", "test"]:
    for cls in ["pizza", "steak"]:
        path = f"/content/pizza_steak/{split}/{cls}"
        n = len(os.listdir(path))
        print(f"  {split}/{cls}: {n} images")

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

tf.random.set_seed(630)
np.random.seed(630)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_DIR = "/content/pizza_steak/train"
TEST_DIR  = "/content/pizza_steak/test"

print("TensorFlow version:", tf.__version__)

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Data Preparation</b></p>
</div>

### Preprocessing Decisions
- **Resizing to 224×224**: EfficientNet and VGG-family models were trained on 224×224 images, so we use the same size for both models to enable a fair comparison.
- **Normalization (÷255)**: Scales pixel values from [0,255] to [0,1], which stabilizes gradient descent and speeds convergence.
- **Data augmentation (train only)**: Horizontal flips, small rotations, and zoom help the model generalize by exposing it to varied presentations of the same food items.
- **No augmentation on test set**: We rescale only, so evaluation reflects real-world performance.


### Generators for EfficientNetB0 (NO rescaling — model handles it internally)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR, batch_size=BATCH_SIZE,
    target_size=IMG_SIZE, class_mode="binary", seed=630
)
test_data = test_datagen.flow_from_directory(
    TEST_DIR, batch_size=BATCH_SIZE,
    target_size=IMG_SIZE, class_mode="binary", seed=630
)


### Generators for EfficientNetB0 (NO rescaling — model handles it internally)

In [ ]:

from tensorflow.keras.applications.efficientnet import preprocess_input

train_datagen_tl = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1
)
test_datagen_tl = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data_tl = train_datagen_tl.flow_from_directory(
    TRAIN_DIR, batch_size=BATCH_SIZE,
    target_size=IMG_SIZE, class_mode="binary", seed=630
)
test_data_tl = test_datagen_tl.flow_from_directory(
    TEST_DIR, batch_size=BATCH_SIZE,
    target_size=IMG_SIZE, class_mode="binary", seed=630
)

### Visualize sample images from the dataset

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

batch_imgs, batch_labels = next(iter(train_data))
class_names = ["pizza", "steak"]  # 0=pizza, 1=steak (alphabetical)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Sample Training Images (after augmentation)", fontsize=14)
for i, ax in enumerate(axes.flatten()):
    ax.imshow(batch_imgs[i])
    ax.set_title(class_names[int(batch_labels[i])], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>CNN Built From Scratch</b></p>
</div>

### Architecture Rationale
We build a 3-block CNN: each block contains a **Conv2D → BatchNorm → ReLU → MaxPool** pattern.

| Layer type | Purpose |
|---|---|
| Conv2D (3×3) | Learn local spatial features (edges, textures, shapes) |
| BatchNormalization | Stabilize training, reduce internal covariate shift |
| ReLU activation | Introduce non-linearity; avoid vanishing gradients |
| MaxPooling2D | Downsample feature maps; provide translation invariance |
| Dropout (0.4) | Regularize; prevent overfitting on small dataset |
| Dense (128) + sigmoid | Binary output (pizza=0, steak=1) |

**Training choices:**
- **Loss**:  — correct loss for binary classification.
- **Optimizer**: Adam (lr=1e-3) — adaptive learning rates, works well out-of-the-box.
- **Early stopping**: Monitor val_loss with patience=5 to avoid overfitting.
- **ReduceLROnPlateau**: Halves learning rate if val_loss stalls for 3 epochs.


In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def build_cnn_from_scratch(input_shape=(224,224,3)):
    model = models.Sequential(name="cnn_scratch")
    model.add(layers.Input(shape=input_shape))

    # Block 1
    model.add(layers.Conv2D(32, (3,3), padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D(2,2))

    # Block 2
    model.add(layers.Conv2D(64, (3,3), padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D(2,2))

    # Block 3
    model.add(layers.Conv2D(128, (3,3), padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D(2,2))

    # Classifier head
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dropout(0.4))
    model.add(layers.Dense(1, activation="sigmoid"))

    return model

scratch_model = build_cnn_from_scratch()
scratch_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
scratch_model.summary()

### Callbacks

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)

### Train CNN From Scratch

In [ ]:
scratch_history = scratch_model.fit(
    train_data,
    epochs=30,
    validation_data=test_data,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("Best val accuracy (scratch CNN):",
      max(scratch_history.history["val_accuracy"]))

### Plot Training Curves For CNN From Scratch

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=13)

    axes[0].plot(history.history["accuracy"],   label="Train Acc")
    axes[0].plot(history.history["val_accuracy"], label="Val Acc")
    axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch")
    axes[0].legend(); axes[0].set_ylim(0, 1)

    axes[1].plot(history.history["loss"],   label="Train Loss")
    axes[1].plot(history.history["val_loss"], label="Val Loss")
    axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(scratch_history, "CNN from Scratch — Training Curves")

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Transfer Learning Model - EfficientNetB0</b></p>
</div>

### Why EfficientNetB0?
EfficientNetB0 was selected because:
- It is pre-trained on **ImageNet** (1.2M images, 1000 classes), so it has already learned rich visual features (edges, textures, shapes, objects) that are useful for food classification.
- It is **computationally lightweight** (5.3M parameters) compared to VGG16 (138M) or ResNet50 (25M), making it practical for this class size and hardware.
- It uses **compound scaling** to balance depth, width, and resolution efficiently.

### Transfer Learning vs. Scratch
A CNN trained from scratch must learn *all* visual features from ~750 training images, which is insufficient for complex image features. Transfer learning **reuses** pre-learned ImageNet features and only re-trains a small classifier head on the pizza/steak task, dramatically reducing required data and training time.

### Two-phase training strategy
1. **Phase 1 — Feature extraction**: Freeze all EfficientNet layers; train only the new classifier head (10 epochs).
2. **Phase 2 — Fine-tuning**: Unfreeze the top 20 layers of EfficientNet; train at very low learning rate to adapt higher-level features to food domain.


In [ ]:
from tensorflow.keras.applications import EfficientNetB0

# Load EfficientNetB0 without the top classification head
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # Freeze base for Phase 1

# Add custom classifier head
inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

tl_model = tf.keras.Model(inputs, outputs, name="efficientnet_transfer")

tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print(f"Trainable params (Phase 1): {tl_model.trainable_variables.__len__()}")
tl_model.summary()

### Phase 1 - Train Only the Head (Feature Extraction)

In [ ]:
early_stop1 = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_phase1 = tl_model.fit(
    train_data_tl,
    epochs=10,
    validation_data=test_data_tl,
    callbacks=[early_stop1],
    verbose=1
)

print("Phase 1 best val accuracy:", max(history_phase1.history["val_accuracy"]))

### Phase 2 - Fine-Tune Top 20 Layers of EfficientNet

In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 20
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # Very low LR
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop2 = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
reduce_lr2   = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-8)

history_phase2 = tl_model.fit(
    train_data_tl,
    epochs=20,
    validation_data=test_data_tl,
    callbacks=[early_stop2, reduce_lr2],
    verbose=1
)

print("Phase 2 best val accuracy:", max(history_phase2.history["val_accuracy"]))

### Combine Training Histories For Plotting

In [ ]:
combined_acc  = history_phase1.history["accuracy"]     + history_phase2.history["accuracy"]
combined_val  = history_phase1.history["val_accuracy"]  + history_phase2.history["val_accuracy"]
combined_loss = history_phase1.history["loss"]         + history_phase2.history["loss"]
combined_vloss= history_phase1.history["val_loss"]     + history_phase2.history["val_loss"]
phase1_end    = len(history_phase1.history["accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("EfficientNetB0 Transfer Learning — Training Curves", fontsize=13)

axes[0].plot(combined_acc,  label="Train Acc")
axes[0].plot(combined_val,  label="Val Acc")
axes[0].axvline(phase1_end, color="gray", linestyle="--", label="Fine-tune start")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].set_ylim(0, 1)

axes[1].plot(combined_loss,  label="Train Loss")
axes[1].plot(combined_vloss, label="Val Loss")
axes[1].axvline(phase1_end, color="gray", linestyle="--", label="Fine-tune start")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Evaluation & Comparison</b></p>
</div>


We evaluate both models on the held-out test set using accuracy, a confusion matrix, and a full classification report.


### Evaluate both models on the test set

In [ ]:
scratch_loss, scratch_acc = scratch_model.evaluate(test_data, verbose=0)
tl_loss, tl_acc = tl_model.evaluate(test_data_tl, verbose=0)

print(f"CNN from Scratch  — Test Accuracy: {scratch_acc:.4f} | Test Loss: {scratch_loss:.4f}")
print(f"EfficientNetB0 TL — Test Accuracy: {tl_acc:.4f}      | Test Loss: {tl_loss:.4f}")

### Summary Tables

In [ ]:
comparison_df = pd.DataFrame({
    "Model":    ["CNN from Scratch", "EfficientNetB0 TL"],
    "Test Acc": [round(scratch_acc,4), round(tl_acc,4)],
    "Test Loss":[round(scratch_loss,4), round(tl_loss,4)]
})
print(" ", comparison_df.to_string(index=False) 

### Helper: Get All Predictions & True Labels From a Generator

In [ ]:
# Helper: get all predictions and true labels from a generator
def get_predictions(model, data_gen):
    data_gen.reset()
    y_true, y_pred_prob = [], []
    for i in range(len(data_gen)):
        imgs, labels = data_gen[i]
        preds = model.predict(imgs, verbose=0).flatten()
        y_pred_prob.extend(preds)
        y_true.extend(labels)
    y_pred = (np.array(y_pred_prob) >= 0.5).astype(int)
    return np.array(y_true), np.array(y_pred), np.array(y_pred_prob)

class_names = list(test_data.class_indices.keys())  # ["pizza", "steak"]

y_true_s, y_pred_s, _ = get_predictions(scratch_model, test_data)
y_true_t, y_pred_t, _ = get_predictions(tl_model, test_data_tl)

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_true, y_pred, title in [
    (axes[0], y_true_s, y_pred_s, "CNN from Scratch"),
    (axes[1], y_true_t, y_pred_t, "EfficientNetB0 TL")
]:
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title)
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")

plt.suptitle("Confusion Matrices — Test Set", fontsize=13)
plt.tight_layout()
plt.show()

### Classification reports

In [ ]:
print("=== CNN from Scratch ===")
print(classification_report(y_true_s, y_pred_s, target_names=class_names))
print("=== EfficientNetB0 Transfer Learning ===")
print(classification_report(y_true_t, y_pred_t, target_names=class_names))

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

# Load test images with file paths for inspection
test_datagen_raw = ImageDataGenerator(preprocessing_function=preprocess_input)
test_data_raw = test_datagen_raw.flow_from_directory(
    TEST_DIR, batch_size=1, target_size=IMG_SIZE,
    class_mode="binary", seed=630, shuffle=False
)

# Collect misclassified images
wrong_imgs, wrong_true, wrong_pred = [], [], []
test_data_raw.reset()

for i in range(len(test_data_raw)):
    img, label = test_data_raw[i]
    prob = tl_model.predict(img, verbose=0)[0][0]
    pred = int(prob >= 0.5)
    true = int(label[0])
    if pred != true:
        wrong_imgs.append(img[0])
        wrong_true.append(true)
        wrong_pred.append(pred)

print(f"Misclassified images (TL model): {len(wrong_imgs)} / {len(test_data_raw)}")

n_show = len(wrong_imgs)  # just 3
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 5))
fig.suptitle("Misclassified Images — EfficientNetB0 TL", fontsize=13)
for i, ax in enumerate(axes):
    ax.imshow(np.clip(wrong_imgs[i], 0, 1))
    ax.set_title(f"True: {class_names[wrong_true[i]]}\nPred: {class_names[wrong_pred[i]]}",
                 color="red", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Error Analysis Discussion

After examining misclassified images, common failure modes include:

- **Ambiguous presentation**: Images showing pizza cut into strips or steak served on pizza-style platters confuse the model.
- **Lighting and angle**: Dark, top-down or heavily shadowed images lose the color and texture cues the model relies on.
- **Close-up textures**: Extreme close-ups of cheese (pizza) or fat marbling (steak) can look similar at the pixel level.
- **Class imbalance / limited data**: With ~750 training images split across two classes, the model has limited exposure to edge cases.

**Deployment implication**: In a real application (e.g., menu-photo tagging), a confidence threshold could be applied — images where the model is uncertain (predicted probability near 0.5) could be flagged for human review rather than auto-labeled.



<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Feature Space Visualization (t-SNE)</b></p>
</div>

We extract the 128-dimensional feature vectors from the penultimate dense layer of the TL model and reduce them to 2D using t-SNE. This shows *how the model "sees" the images* — well-separated clusters indicate the model has learned distinguishing features.


In [ ]:
# Create a feature extractor model (output of the Dense-128 layer)
feature_model = tf.keras.Model(
    inputs=tl_model.input,
    outputs=tl_model.layers[-3].output  # Dense(128) layer
)

# Extract features from all test images
test_datagen_fe = ImageDataGenerator(preprocessing_function=preprocess_input)
test_data_fe = test_datagen_fe.flow_from_directory(
    TEST_DIR, batch_size=BATCH_SIZE, target_size=IMG_SIZE,
    class_mode="binary", seed=630, shuffle=False
)

features, labels_fe = [], []
test_data_fe.reset()
for i in range(len(test_data_fe)):
    imgs, lbls = test_data_fe[i]
    feats = feature_model.predict(imgs, verbose=0)
    features.extend(feats)
    labels_fe.extend(lbls)

features   = np.array(features)
labels_fe  = np.array(labels_fe).astype(int)
print(f"Feature matrix shape: {features.shape}")

In [ ]:
# t-SNE dimensionality reduction
tsne = TSNE(n_components=2, random_state=630, perplexity=30, n_iter=1000)
features_2d = tsne.fit_transform(features)

# Get predictions to color correctly/incorrectly classified
test_data_fe.reset()
pred_probs_fe = tl_model.predict(test_data_fe, verbose=0).flatten()
pred_labels_fe = (pred_probs_fe >= 0.5).astype(int)
correct_mask = (pred_labels_fe == labels_fe)

# Plot 1: True class labels
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("t-SNE Feature Space — EfficientNetB0 TL (Test Set)", fontsize=13)

colors = ["#e74c3c" if l == 0 else "#2980b9" for l in labels_fe]
axes[0].scatter(features_2d[:,0], features_2d[:,1],
               c=colors, alpha=0.7, s=40, edgecolors="k", linewidths=0.3)
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color="#e74c3c", label="Pizza"),
    Patch(color="#2980b9", label="Steak")
], loc="best")
axes[0].set_title("Colored by True Class")
axes[0].set_xlabel("t-SNE Dim 1"); axes[0].set_ylabel("t-SNE Dim 2")

# Plot 2: Correct vs. Incorrect predictions
colors2 = ["green" if c else "red" for c in correct_mask]
axes[1].scatter(features_2d[:,0], features_2d[:,1],
               c=colors2, alpha=0.7, s=40, edgecolors="k", linewidths=0.3)
axes[1].legend(handles=[
    Patch(color="green", label="Correct"),
    Patch(color="red",   label="Misclassified")
], loc="best")
axes[1].set_title("Colored by Prediction Correctness")
axes[1].set_xlabel("t-SNE Dim 1")

plt.tight_layout()
plt.show()

print("Interpretation: Well-separated red/blue clusters indicate the model has learned")
print("discriminative features. Misclassified points (red in right plot) tend to appear")
print("near the boundary between the two class clusters.")

<div style="color:white;display:fill;
            background-color:#3bb260;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Executive Recommendation</b></p>
</div>


After testing two image classification approaches on a dataset of pizza and steak photos, the transfer learning model built on EfficientNetB0 is the clear recommendation for business deployment. It correctly identified 497 out of 500 test images for an accuracy of 99.4%, compared to 92.4% for the custom-built model trained from scratch. Beyond raw accuracy, the transfer learning model trained faster, converged more reliably, and made far fewer mistakes — only 3 misclassifications versus 38 for the scratch model.

The practical case for the transfer learning model is straightforward. It leverages a neural network already trained on over a million images, meaning it arrives pre-equipped with the ability to recognize edges, textures, and shapes before ever seeing a single food photo. This makes it both more accurate and more efficient than building a model from the ground up on a small dataset. For a business deploying this in a real application such as menu photo organization or food delivery categorization, the transfer learning model offers a level of reliability that is production-ready, with the added option of flagging the rare uncertain prediction for human review rather than auto-labeling it.

